# KV-aware routing vs random routing on Amazon SageMaker AI

KV-aware routing (`RoutingStrategy: KV_AWARE`) sends each request to the instance whose GPU already holds the longest matching part of the prompt in its KV cache. This notebook measures what that is worth.

**What it does.** It deploys the same model twice, one time with `RANDOM` routing and one time with `KV_AWARE` routing. It sends the same requests to both and compares the Amazon CloudWatch metrics of the two endpoints.

**Steps**

| Step | Section |
|---|---|
| 1 | Set up, and build one set of requests that both endpoints will receive |
| 2 | Deploy `RANDOM`, send the load, read CloudWatch, delete the endpoint |
| 3 | Deploy `KV_AWARE`, send the same load, read CloudWatch, delete the endpoint |
| 4 | Compare the two |

**Why one at a time.** Only one endpoint exists at any moment, so the test needs 2 GPU instances, not 4. Both endpoints start with empty caches, which is the fair starting point.

**What stays identical between the two runs.** The container image, model, environment variables, instance type, instance count, and every request body. The notebook hashes the request bodies and checks that both runs sent the same hash. The only setting that differs is `RoutingStrategy`.

**This notebook uses a single model endpoint** to keep the comparison simple. `KV_AWARE` routing also works with inference component endpoints.

**Cost.** 2 x `ml.g5.xlarge` while an endpoint exists. Endpoint start-up time varies, so watch the progress dots. The load test itself takes a few minutes.

In [ ]:
%pip install --upgrade --quiet --no-warn-conflicts boto3 pandas numpy

In [ ]:
import hashlib
import json
import math
import random
import re
import threading
import time
from datetime import datetime, timedelta, timezone

import boto3
import numpy as np
import pandas as pd
from botocore.config import Config
from botocore.exceptions import ClientError
from IPython.display import clear_output, display

boto_session = boto3.Session()
region = boto_session.region_name

sm = boto3.client("sagemaker")
sm_runtime = boto3.client("sagemaker-runtime",
                          config=Config(read_timeout=300, retries={"max_attempts": 1}, max_pool_connections=128))
cw = boto3.client("cloudwatch")
print("Region:", region)

In [ ]:
#
# Helper functions to remove dependency on SageMaker Python SDK
#
def get_sagemaker_role():
    arn = boto3.client("sts").get_caller_identity()["Arn"]
    return re.sub(r"^(.+)sts::(\d+):assumed-role/(.+?)/.*$", r"\1iam::\2:role/\3", arn)


def wait_for_endpoint(endpoint_name: str, sleep_time: int = 30):
    """Poll until the endpoint leaves Creating/Updating. Raises if it ends up Failed."""
    progress = ""
    while True:
        d = sm.describe_endpoint(EndpointName=endpoint_name)
        status = d["EndpointStatus"]
        if status not in ("Creating", "Updating"):
            break
        progress += "."
        clear_output(wait=True)
        print(f"Waiting for '{endpoint_name}': {progress}")
        time.sleep(sleep_time)
    print(f"Endpoint: '{endpoint_name}', Status: '{status}'")
    if status != "InService":
        raise RuntimeError(f"{endpoint_name} is {status}: {d.get('FailureReason')}")

In [ ]:
#
# Overwrite with your role ARN if you are running this notebook outside of SageMaker Studio
#
role = None

if role is None:
    role = get_sagemaker_role()
print(role)

## 1. Configuration

Everything in this cell is shared by both deployments. Change it here, not in the deployment cells, so that both runs stay identical.

**Model size matters for what you will see.** A very small model, such as one with 0.5B parameters, reads a long prompt so fast that skipping it saves little time. This notebook uses a 4B model on an A10G GPU, where skipping a 7,400-token prompt is clearly visible. You can change `model_id` and `instance` in the next cell.

**Region.** Use a Region where the instance type is available and you have endpoint quota for it.

In [ ]:
instance = {"type": "ml.g5.xlarge", "num_gpu": 1}   # 1 x NVIDIA A10G, 24 GB
instance_count = 2                                   # routing needs at least 2 instances to have a choice

model_id = "Qwen/Qwen3-4B-Instruct-2507"             # open model, no Hugging Face token needed

inference_image = f"763104351884.dkr.ecr.{region}.amazonaws.com/vllm:0.33.0-gpu-py312-cu130-ubuntu24.04-sagemaker"
ami_version = "al2023-ami-sagemaker-inference-gpu-4-1"

run_id = time.strftime("%y%m%d-%H%M%S")
model_name = f"kvab-model-{run_id}"
variant_name = "v1"
timeout = 1500            # seconds allowed for model download and container start

kv_events_config = {
    "enable_kv_cache_events": True,
    "publisher": "zmq",
    "endpoint": "tcp://*:5557",
    "topic": "kv-events",
    "replay_endpoint": "tcp://*:5658",
}

# The same environment is used for both endpoints. RANDOM ignores the KV events; leaving them on keeps the
# container identical, so routing is the only difference.
env = {
    "SM_VLLM_MODEL": model_id,
    "SM_VLLM_TENSOR_PARALLEL_SIZE": json.dumps(instance["num_gpu"]),
    "SM_VLLM_MAX_MODEL_LEN": "16384",
    "SM_VLLM_GPU_MEMORY_UTILIZATION": "0.90",
    "SM_VLLM_ENABLE_PROMPT_TOKENS_DETAILS": "true",   # responses report usage.prompt_tokens_details.cached_tokens
    "SM_VLLM_KV_EVENTS_CONFIG": json.dumps(kv_events_config),
}
tags = [{"Key": "Project", "Value": "kv-aware-routing-ab"}]

## 2. Deployment helpers

`deploy_endpoint(strategy)` creates the model (once), an endpoint configuration and the endpoint. The routing strategy is the only argument. Detailed observability is turned on for both endpoints so that both publish the same set of metrics.

In [ ]:
def ensure_model():
    try:
        sm.describe_model(ModelName=model_name)
    except ClientError:
        sm.create_model(
            ModelName=model_name,
            ExecutionRoleArn=role,
            PrimaryContainer={"Image": inference_image, "Environment": env},
            Tags=tags,
        )
        print("Created model", model_name)


def deploy_endpoint(strategy: str) -> str:
    ensure_model()
    name = f"kvab-{strategy.lower().replace('_', '-')}-{run_id}"
    sm.create_endpoint_config(
        EndpointConfigName=name,
        MetricsConfig={"EnableDetailedObservability": True},
        ProductionVariants=[{
            "VariantName": variant_name,
            "ModelName": model_name,
            "InstanceType": instance["type"],
            "InitialInstanceCount": instance_count,
            "ContainerStartupHealthCheckTimeoutInSeconds": timeout,
            "ModelDataDownloadTimeoutInSeconds": timeout,
            "InferenceAmiVersion": ami_version,
            "RoutingConfig": {"RoutingStrategy": strategy},
        }],
        Tags=tags,
    )
    sm.create_endpoint(EndpointName=name, EndpointConfigName=name, Tags=tags)
    wait_for_endpoint(name)
    return name


def delete_endpoint(name: str):
    sm.delete_endpoint(EndpointName=name)
    sm.delete_endpoint_config(EndpointConfigName=name)
    print("Deleted endpoint and endpoint config", name)

## 3. The load: one set of requests for both endpoints

**What the load looks like.** Each simulated user picks one of a small set of long documents and asks several questions about it in a conversation. Every question resends the whole conversation so far. Questions 2 and later therefore start with text that an instance may already have in its KV cache: the document, and the earlier questions. Several users share the same document.

**Why the requests are identical.** The assistant turns in each conversation are a fixed sentence and not the model's real answer. If real answers were used, the two endpoints would build slightly different histories. The request bodies are built once, before any endpoint exists, and hashed. The hash of what each run sent is printed and checked.

**What is measured on the client.** Time to first token (TTFT): the time from sending a request until the first piece of the answer arrives. Also the share of prompt tokens that the container reports as served from cache.

In [ ]:
workload = dict(
    users=32,          # simultaneous conversations
    rounds=4,          # questions per conversation
    docs=12,           # distinct documents; users share them
    doc_words=2200,    # about 7,400 tokens per document with the Qwen3 tokenizer
    max_tokens=48,     # short answers keep the test about prompt processing
    ramp_s=0.5,        # seconds between user starts
    doc_seed=7000,     # change to get new documents
)

_WORDS = ("amber basil cedar delta ember fjord garnet harbor indigo juniper kestrel lantern meadow nectar orchid "
          "pebble quartz river saffron timber umber velvet willow xenon yarrow zephyr anchor bridge candle dune "
          "engine forest glacier hollow island jungle kettle ladder marble nickel oasis prairie quiver ridge "
          "summit tunnel upland valley wagon yonder zenith atlas beacon canyon drift echo flint grove haven").split()
_SYSTEM = "You are a careful research assistant. Answer questions about the document below in one or two sentences."
_QUESTIONS = ["What is the first word of the document?", "Name one word from the middle of the document.",
              "Name one word near the end of the document.", "Does the document mention a river?",
              "Does the document mention a lantern?", "Name two colors that appear in the document.",
              "How would you describe the style of the document?", "Name a place word from the document."]
_FIXED_ANSWER = "Understood. Here is a short answer based on the document."


def _make_doc(i, words):
    rng = random.Random(i)
    return f"DOCUMENT {i}: " + " ".join(rng.choice(_WORDS) + str(rng.randint(0, 99)) for _ in range(words))


def build_requests(w):
    """Returns one list per user; each list holds that user's request bodies in order."""
    docs = [_make_doc(w["doc_seed"] + i, w["doc_words"]) for i in range(w["docs"])]
    per_user = []
    for u in range(w["users"]):
        messages = [{"role": "system", "content": _SYSTEM + "\n\n" + docs[u % w["docs"]]}]
        bodies = []
        for r in range(w["rounds"]):
            messages.append({"role": "user", "content": _QUESTIONS[(u + r) % len(_QUESTIONS)]})
            bodies.append(json.dumps({"messages": messages, "max_tokens": w["max_tokens"], "temperature": 0,
                                      "stream": True, "stream_options": {"include_usage": True}},
                                     separators=(",", ":")))
            messages = messages + [{"role": "assistant", "content": _FIXED_ANSWER}]
        per_user.append(bodies)
    return per_user


def fingerprint(per_user):
    h = hashlib.sha256()
    for bodies in per_user:
        for b in bodies:
            h.update(b.encode())
    return h.hexdigest()[:16]


requests_by_user = build_requests(workload)
n_requests = sum(len(b) for b in requests_by_user)
workload_hash = fingerprint(requests_by_user)
avg_kb = np.mean([len(b) for u in requests_by_user for b in u]) / 1024
print(f"{workload['users']} users x {workload['rounds']} questions = {n_requests} requests")
print(f"Average request body: {avg_kb:.0f} KiB")
print(f"Workload hash: {workload_hash}")

In [ ]:
def _one_request(endpoint_name, body):
    t0 = time.perf_counter()
    ttft, usage, err = None, None, None
    try:
        resp = sm_runtime.invoke_endpoint_with_response_stream(
            EndpointName=endpoint_name, ContentType="application/json", Body=body)
        buf = b""
        for ev in resp["Body"]:
            buf += ev.get("PayloadPart", {}).get("Bytes", b"")
            while b"\n" in buf:
                line, buf = buf.split(b"\n", 1)
                line = line.strip()
                if not line.startswith(b"data:") or line.endswith(b"[DONE]"):
                    continue
                chunk = json.loads(line[5:])
                if chunk.get("usage"):
                    usage = chunk["usage"]
                for choice in chunk.get("choices", []):
                    if choice.get("delta", {}).get("content") and ttft is None:
                        ttft = time.perf_counter() - t0
    except Exception as e:  # noqa: BLE001 - record and keep the test going
        err = str(e)[:200]
    details = (usage or {}).get("prompt_tokens_details") or {}
    return dict(ttft_ms=None if ttft is None else ttft * 1000,
                total_ms=(time.perf_counter() - t0) * 1000,
                prompt_tokens=(usage or {}).get("prompt_tokens"),
                cached_tokens=details.get("cached_tokens"),
                error=err)


def run_load_test(endpoint_name, per_user, ramp_s):
    """Each user sends its requests one after another; users start ramp_s apart."""
    rows, lock, threads = [], threading.Lock(), []

    def user(uid, bodies):
        for r, body in enumerate(bodies, start=1):
            res = _one_request(endpoint_name, body)
            res.update(user=uid, round=r)
            with lock:
                rows.append(res)

    start = datetime.now(timezone.utc)
    t0 = time.time()
    for uid, bodies in enumerate(per_user):
        t = threading.Thread(target=user, args=(uid, bodies))
        t.start()
        threads.append(t)
        time.sleep(ramp_s)
    for t in threads:
        t.join()
    end = datetime.now(timezone.utc)
    return dict(df=pd.DataFrame(rows), start=start, end=end, wall_s=time.time() - t0,
                workload_hash=fingerprint(per_user))


def summarize_client(run):
    df = run["df"]
    ok = df[df["error"].isna() & df["ttft_ms"].notna()]
    later = ok[ok["round"] > 1]
    with_cache = ok[ok["cached_tokens"].notna() & ok["prompt_tokens"].notna()]
    return {
        "Requests sent": len(df),
        "Requests failed": int(df["error"].notna().sum()),
        "Run duration (s)": round(run["wall_s"], 1),
        "TTFT p50, all requests (ms)": round(ok["ttft_ms"].quantile(0.50)),
        "TTFT p90, all requests (ms)": round(ok["ttft_ms"].quantile(0.90)),
        "TTFT p50, question 2+ (ms)": round(later["ttft_ms"].quantile(0.50)),
        "TTFT p90, question 2+ (ms)": round(later["ttft_ms"].quantile(0.90)),
        "Prompt tokens served from cache (share)": round(with_cache["cached_tokens"].sum() / with_cache["prompt_tokens"].sum(), 3),
    }

## 4. CloudWatch metrics

`collect_cloudwatch(endpoint, start, end, expected)` waits until CloudWatch has received the invocations of the run, then reads these metrics for the time window of the load test:

| Metric | Namespace | Meaning |
|---|---|---|
| `Invocations` | `AWS/SageMaker` | Requests the endpoint received |
| `Invocation4XXErrors`, `Invocation5XXErrors` | `AWS/SageMaker` | Requests that failed |
| `ModelLatency` | `AWS/SageMaker` | Time the model container took to respond, in microseconds. This notebook shows it in milliseconds |
| `OverheadLatency` | `AWS/SageMaker` | Time SageMaker spent outside the model container |
| `GPUUtilization`, `GPUMemoryUtilization` | `/aws/sagemaker/Endpoints` | GPU busy time and GPU memory use, in percent |

CloudWatch publishes these metrics once a minute and delays them by a minute or more, so the function can wait several minutes. The metrics stay in CloudWatch after the endpoint is deleted.

In [ ]:
def _metric_query(qid, namespace, metric, stat, dims, period):
    return {"Id": qid, "ReturnData": True, "MetricStat": {
        "Metric": {"Namespace": namespace, "MetricName": metric,
                   "Dimensions": [{"Name": k, "Value": v} for k, v in dims.items()]},
        "Period": period, "Stat": stat}}


def _read(queries, start, end):
    out = {}
    token = None
    while True:
        kwargs = dict(MetricDataQueries=queries, StartTime=start, EndTime=end, ScanBy="TimestampAscending")
        if token:
            kwargs["NextToken"] = token
        r = cw.get_metric_data(**kwargs)
        for res in r["MetricDataResults"]:
            out.setdefault(res["Id"], []).extend(res["Values"])
        token = r.get("NextToken")
        if not token:
            return out


def collect_cloudwatch(endpoint_name, start, end, expected, max_wait_s=900):
    """One value per metric for the whole load-test window (one CloudWatch period covers the window)."""
    win_start = start.replace(second=0, microsecond=0) - timedelta(minutes=1)
    win_end = end + timedelta(minutes=3)
    period = int(math.ceil((win_end - win_start).total_seconds() / 60.0) * 60)

    def mq(qid, namespace, metric, stat, dims):
        return _metric_query(qid, namespace, metric, stat, dims, period)

    ep = {"EndpointName": endpoint_name, "VariantName": variant_name}
    sm_ns, gpu_ns = "AWS/SageMaker", "/aws/sagemaker/Endpoints"

    deadline = time.time() + max_wait_s
    while True:
        got = _read([mq("inv", sm_ns, "Invocations", "Sum", ep)], win_start, win_end).get("inv", [])
        seen = sum(got)
        if seen >= expected or time.time() > deadline:
            break
        clear_output(wait=True)
        print(f"Waiting for CloudWatch: {int(seen)} of {expected} invocations visible")
        time.sleep(30)
    if seen < expected:
        print(f"WARNING: only {int(seen)} of {expected} invocations visible; numbers below may be incomplete")

    q = [
        mq("inv", sm_ns, "Invocations", "Sum", ep),
        mq("e4", sm_ns, "Invocation4XXErrors", "Sum", ep),
        mq("e5", sm_ns, "Invocation5XXErrors", "Sum", ep),
        mq("ml_avg", sm_ns, "ModelLatency", "Average", ep),
        mq("ml_p50", sm_ns, "ModelLatency", "p50", ep),
        mq("ml_p90", sm_ns, "ModelLatency", "p90", ep),
        mq("ml_p99", sm_ns, "ModelLatency", "p99", ep),
        mq("oh_avg", sm_ns, "OverheadLatency", "Average", ep),
        mq("oh_p90", sm_ns, "OverheadLatency", "p90", ep),
        mq("gpu_avg", gpu_ns, "GPUUtilization", "Average", ep),
        mq("gpu_max", gpu_ns, "GPUUtilization", "Maximum", ep),
        mq("gmem_avg", gpu_ns, "GPUMemoryUtilization", "Average", ep),
        mq("gmem_max", gpu_ns, "GPUMemoryUtilization", "Maximum", ep),
    ]
    r = _read(q, win_start, win_end)

    def one(k, scale=1.0):
        v = r.get(k, [])
        return None if not v else round(v[0] * scale, 1) if len(v) == 1 else round(float(np.mean(v)) * scale, 1)

    metrics = {
        "Invocations": int(sum(r.get("inv", [0]))),
        "Invocation4XXErrors": int(sum(r.get("e4", [0]))),
        "Invocation5XXErrors": int(sum(r.get("e5", [0]))),
        "ModelLatency average (ms)": one("ml_avg", 0.001),
        "ModelLatency p50 (ms)": one("ml_p50", 0.001),
        "ModelLatency p90 (ms)": one("ml_p90", 0.001),
        "ModelLatency p99 (ms)": one("ml_p99", 0.001),
        "OverheadLatency average (ms)": one("oh_avg", 0.001),
        "OverheadLatency p90 (ms)": one("oh_p90", 0.001),
        "GPUUtilization average (%)": one("gpu_avg"),
        "GPUUtilization maximum (%)": one("gpu_max"),
        "GPUMemoryUtilization average (%)": one("gmem_avg"),
        "GPUMemoryUtilization maximum (%)": one("gmem_max"),
    }

    return metrics

## 5. Run A: `RANDOM` routing

### 5.1 Deploy

In [ ]:
results = {}                      # filled in by the two runs
ep_random = deploy_endpoint("RANDOM")

### 5.2 Send the load

In [ ]:
# One request to check the endpoint answers. It puts one short prompt into one cache, which does not affect the test.
r = sm_runtime.invoke_endpoint(EndpointName=ep_random, ContentType="application/json",
                               Body=json.dumps({"messages": [{"role": "user", "content": "Who are you?"}], "max_tokens": 32}))
print(json.loads(r["Body"].read())["choices"][0]["message"]["content"])

run_random = run_load_test(ep_random, requests_by_user, workload["ramp_s"])
assert run_random["workload_hash"] == workload_hash
print("Workload hash sent:", run_random["workload_hash"])
results["RANDOM"] = dict(endpoint=ep_random, client=summarize_client(run_random))
display(pd.Series(results["RANDOM"]["client"], name="RANDOM (client side)").to_frame())

### 5.3 CloudWatch metrics

In [ ]:
results["RANDOM"]["cloudwatch"] = collect_cloudwatch(ep_random, run_random["start"], run_random["end"], expected=n_requests + 1)
display(pd.Series(results["RANDOM"]["cloudwatch"], name="RANDOM (CloudWatch)").to_frame())

### 5.4 Delete the `RANDOM` endpoint

This stops the billing for these 2 GPU instances and frees the quota for the next run.

In [ ]:
delete_endpoint(ep_random)

## 6. Run B: `KV_AWARE` routing

The cells repeat Run A. The only change is the strategy name.

### 6.1 Deploy

In [ ]:
ep_kv = deploy_endpoint("KV_AWARE")

### 6.2 Send the same load

In [ ]:
r = sm_runtime.invoke_endpoint(EndpointName=ep_kv, ContentType="application/json",
                               Body=json.dumps({"messages": [{"role": "user", "content": "Who are you?"}], "max_tokens": 32}))
print(json.loads(r["Body"].read())["choices"][0]["message"]["content"])

run_kv = run_load_test(ep_kv, requests_by_user, workload["ramp_s"])
assert run_kv["workload_hash"] == workload_hash == run_random["workload_hash"], "the two runs sent different requests"
print("Workload hash sent:", run_kv["workload_hash"], "(same as the RANDOM run)")
results["KV_AWARE"] = dict(endpoint=ep_kv, client=summarize_client(run_kv))
display(pd.Series(results["KV_AWARE"]["client"], name="KV_AWARE (client side)").to_frame())

### 6.3 CloudWatch metrics

In [ ]:
results["KV_AWARE"]["cloudwatch"] = collect_cloudwatch(ep_kv, run_kv["start"], run_kv["end"], expected=n_requests + 1)
display(pd.Series(results["KV_AWARE"]["cloudwatch"], name="KV_AWARE (CloudWatch)").to_frame())

### 6.4 Delete the `KV_AWARE` endpoint

In [ ]:
delete_endpoint(ep_kv)

## 7. Compare

Each table has one row per metric. The last column is the change from `RANDOM` to `KV_AWARE`, in percent. For latency and error rows, a negative number means `KV_AWARE` is lower.

**How to read it**

* **CloudWatch table** is what the service reports for each endpoint. It is the primary comparison.
* **Client table** is measured by this notebook. TTFT (time to first token) is the number a user of a chat application feels. CloudWatch has no first-token metric.
* **Prompt tokens served from cache** (client table) is the direct evidence of routing: it is the share of prompt tokens that an instance did not have to process again.
* **GPU metrics** are sampled once a minute. A load test of a few minutes gives only a few samples, so treat them as rough.
* **Run-to-run variation is not measured by one pair of runs.** To judge whether a difference is larger than noise, repeat sections 5 and 6 and compare the pairs. Use a new `doc_seed` for each repeat, so that no run reuses documents from an earlier run.

In [ ]:
def compare(kind):
    a = pd.Series(results["RANDOM"][kind], name="RANDOM")
    b = pd.Series(results["KV_AWARE"][kind], name="KV_AWARE")
    t = pd.concat([a, b], axis=1)

    def change(row):
        x, y = row["RANDOM"], row["KV_AWARE"]
        if isinstance(x, (int, float)) and isinstance(y, (int, float)) and x:
            return f"{(y - x) / x * 100:+.0f}%"
        return ""
    t["Change"] = t.apply(change, axis=1)
    return t


print("CloudWatch metrics")
display(compare("cloudwatch"))
print("Client-side metrics")
display(compare("client"))

## 8. Cleanup

The endpoints were deleted in sections 5.4 and 6.4. This cell deletes the model. If a run stopped early, it also removes any endpoint of this notebook that is left.

In [ ]:
for name in (globals().get("ep_random"), globals().get("ep_kv")):
    if name:
        try:
            delete_endpoint(name)
        except ClientError:
            pass            # already deleted
sm.delete_model(ModelName=model_name)
print("Deleted model", model_name)